In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
from scipy.stats import chisquare, norm

folder = "data"

files = {
    "Linux /dev/urandom": "data/urandom.bin",
    "OpenSSL rand": "data/openssl.bin",
    "Windows CNG": "data/windows_cng.bin"
}

data = {}

for name, path in files.items():
    if not os.path.exists(path):
        print("Missing file:", path)
    else:
        with open(path, "rb") as f:
            data[name] = np.frombuffer(f.read(), dtype=np.uint8)
        print(name, ":", len(data[name]), "bytes")
        

In [ ]:
plt.figure(figsize=(12, 5))

for name, values in data.items():
    counts = np.bincount(values, minlength=256)
    plt.plot(range(256), counts, label=name)

plt.axhline(1048576 / 256, linestyle="--", label="Expected frequency")
plt.xlabel("Byte value (0–255)")
plt.ylabel("Frequency")
plt.title("Byte Frequency Distribution")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
bit_results = {}

for name, values in data.items():
    bits = np.unpackbits(values)
    zeros = int(np.sum(bits == 0))
    ones = int(np.sum(bits == 1))

    bit_results[name] = {
        "Zero bits": zeros,
        "One bits": ones,
        "One bit percentage": round(100 * ones / len(bits), 4)
    }

pd.DataFrame(bit_results).T

In [ ]:
pd.DataFrame(bit_results).T[["Zero bits", "One bits"]].plot(
    kind="bar", figsize=(9, 5)
)

plt.title("Zero Bits vs One Bits")
plt.ylabel("Number of bits")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
pattern_results = {}

for name, values in data.items():
    adjacent = int(np.sum(values[1:] == values[:-1]))

    sample = values[:400000].tobytes()
    chunks = [sample[i:i+4] for i in range(0, len(sample)-3, 4)]
    counts = Counter(chunks)

    repeated = sum(count - 1 for count in counts.values() if count > 1)

    pattern_results[name] = {
        "Adjacent identical bytes": adjacent,
        "Repeated 4-byte patterns": repeated
    }

pd.DataFrame(pattern_results).T

In [ ]:
correlation_results = {}

for name, values in data.items():
    correlation = np.corrcoef(
        values[:-1].astype(float),
        values[1:].astype(float)
    )[0, 1]

    correlation_results[name] = correlation

pd.DataFrame.from_dict(
    correlation_results,
    orient="index",
    columns=["Serial Correlation"]
)

In [ ]:
test_results = []

for name, values in data.items():
    byte_counts = np.bincount(values, minlength=256)
    expected = np.full(256, len(values) / 256)

    chi_stat, chi_p = chisquare(byte_counts, f_exp=expected)

    bits = np.unpackbits(values).astype(np.int8)
    total = len(bits)
    ones = int(bits.sum())
    proportion = ones / total

    mono_z = (ones - total / 2) / np.sqrt(total / 4)
    mono_p = 2 * norm.sf(abs(mono_z))

    runs = 1 + int(np.sum(bits[1:] != bits[:-1]))
    expected_runs = 1 + 2 * total * proportion * (1 - proportion)

    variance_runs = (
        2 * total * proportion * (1 - proportion)
        * (2 * total * proportion * (1 - proportion) - 1)
        / (total - 1)
    )

    runs_z = (runs - expected_runs) / np.sqrt(variance_runs)
    runs_p = 2 * norm.sf(abs(runs_z))

    test_results.append({
        "Generator": name,
        "Chi-square p-value": chi_p,
        "Bit-balance p-value": mono_p,
        "Runs p-value": runs_p,
        "Number of runs": runs
    })

results = pd.DataFrame(test_results)
results

In [ ]:
summary = []

for name, values in data.items():
    bits = np.unpackbits(values)
    byte_counts = np.bincount(values, minlength=256)

    summary.append({
        "Generator": name,
        "File size (bytes)": len(values),
        "Unique byte values": int(np.count_nonzero(byte_counts)),
        "One bit percentage": round(100 * bits.mean(), 4),
        "Adjacent identical bytes": int(np.sum(values[1:] == values[:-1])),
        "Serial correlation": round(
            np.corrcoef(
                values[:-1].astype(float),
                values[1:].astype(float)
            )[0, 1], 6
        )
    })

summary_df = pd.DataFrame(summary)
summary_df

## Results and Conclusion

The experiment generated 1 MiB of random data using Linux `/dev/urandom`, OpenSSL `rand`, and the Windows CNG/.NET cryptographic random-number API.

The byte-frequency and bit-distribution analyses were used to check whether values were reasonably balanced. Repeated-pattern analysis examined duplicate sequences, while serial correlation measured relationships between consecutive bytes. The chi-square, bit-balance, and runs tests provided additional statistical checks.

Small fluctuations and occasional repeated patterns are expected in random data. Statistical tests can identify certain weaknesses, but they cannot establish that a generator is cryptographically secure.

Cryptographic security also depends on the generator's design, entropy sources, seed handling, internal state protection, and resistance to prediction. Therefore, operating-system cryptographic random-number APIs should be preferred over ordinary non-cryptographic pseudorandom generators for security-sensitive applications.